# v0.18.0 — Field aliases, `server_fields` and `merge(refresh=False)`

Three developer-experience features that let a model's **Python surface** differ from its
**SurrealDB column surface**, and let a caller skip work they do not need:

1. **Field aliases** — `Field(alias="password_hash")` renames the *column*, not the attribute,
   and the ORM honours it on writes, on reads, and in every QuerySet clause.
2. **`server_fields`** — mark the columns the server owns so the client never volunteers them.
3. **`merge(refresh=False)`** — skip the post-merge resync round-trip.

All three are client-side (Pydantic aliasing, payload exclusion, a skipped round-trip), so they
behave **identically on SurrealDB 2.6.x and 3.x**. There is no capability probe in this
notebook and nothing here degrades on an older server — unlike, say, the v0.9.0 interactive
transactions notebook.

## 1. Connect

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. A model whose columns are named differently

Declaration is plain Pydantic — the ORM adds no new syntax. Here the table stores
`password_hash` and `display_name`, while the code reads `password` and `display`.

In [2]:
from pydantic import Field

from surreal_orm_lite import BaseSurrealModel


class Account(BaseSurrealModel):
    id: str
    password: str = Field(default="", alias="password_hash")
    display: str = Field(default="", alias="display_name")
    logins: int = 0


print("alias map:", Account.get_field_aliases())
print("to_db_field('password'):", Account.to_db_field("password"))
print("to_py_field('display_name'):", Account.to_py_field("display_name"))

alias map: {'password': 'password_hash', 'display': 'display_name'}
to_db_field('password'): password_hash
to_py_field('display_name'): display


### Either name validates

`BaseSurrealModel` sets `populate_by_name=True`, so you can build an instance with the Python
name in your own code and with the alias when a row comes back from the server. Before v0.18.0
only the alias was accepted, which made `Account(password=...)` an error in the very code the
alias exists to keep readable.

In [3]:
print(Account(id="a", password="secret"))        # Python name
print(Account(id="b", password_hash="secret"))   # alias

id='a' password='secret' display='' logins=0
id='b' password='secret' display='' logins=0


## 3. Writes store the column, reads hydrate the attribute

We clean the table first, save an instance, then read the row back **raw** — bypassing the
ORM — to see what the server actually stored.

In [4]:
client = await SurrealDBConnectionManager.get_client()


async def raw_row(table: str, record: str) -> dict:
    """Read a row as the server stores it, with no ORM hydration in the way."""
    rows = await client.query(f"SELECT * FROM {table}:{record};", {})
    return dict(rows[0]) if rows else {}


await client.query("REMOVE TABLE IF EXISTS Account;", {})

ada = await Account(id="ada", password="secret", display="Ada").save()
print("stored by the server:", await raw_row("Account", "ada"))

stored by the server: {'display_name': 'Ada', 'id': RecordID(table_name=Account, record_id='ada'), 'logins': 0, 'password_hash': 'secret'}


In [5]:
# ...and the same record read back through the ORM exposes the Python names.
loaded = (await Account.objects().filter(id="ada").exec())[0]
print("password:", loaded.password)
print("display: ", loaded.display)

password: secret
display:  Ada


## 4. Query in Python names — the ORM emits the columns

This is the part the full ORM still documents as an open gap: there, filtering by the Python
name on an aliased model emits the wrong column. In lite the translation covers `select`,
`filter` (including nested and negated `Q` objects), `order_by`, `values`, `fetch`,
`bulk_update` and the `sum`/`avg`/`min`/`max` helpers.

In [6]:
from surreal_orm_lite import Q

query, _ = Account.objects().filter(password="secret").order_by("-display")._compile_query()
print(query)

query, _ = Account.objects().filter(Q(password="a") | Q(display="b"))._compile_query()
print(query)

SELECT * FROM Account WHERE password_hash = $_f0 ORDER BY display_name DESC;
SELECT * FROM Account WHERE (password_hash = $_f0 OR display_name = $_f1);


In [7]:
await Account(id="bob", password="beta", display="Bob", logins=7).save()

found = await Account.objects().filter(display="Bob").exec()
print("found by Python name:", found[0].id, "->", found[0].password)

ordered = await Account.objects().order_by("-password").exec()
print("ordered by Python name:", [item.password for item in ordered])

print("max logins:", await Account.objects().max("logins"))

found by Python name: bob -> beta
ordered by Python name: ['secret', 'beta']
max logins: 7


In [8]:
# bulk_update writes the column too.
count = await Account.objects().filter(password="beta").bulk_update(password="rotated")
print("rows updated:", count)
print("stored:", (await raw_row("Account", "bob"))["password_hash"])

rows updated: 1
stored: rotated


> **One deliberate exception**: `patch()` takes raw RFC 6902 JSON pointers, which address the
> stored document rather than the model — so you write `/password_hash` there, not `/password`.

In [9]:
await ada.patch([{"op": "replace", "path": "/password_hash", "value": "patched"}])
print("after patch:", ada.password)

after patch: patched


## 5. `server_fields` — columns the server owns

`created_at` below is filled by a server-side `DEFAULT`. Marking it in `server_fields` keeps the
client from volunteering it, so the default actually gets to apply instead of being overwritten
with the model's `None`.

In [10]:
from typing import Any

from surreal_orm_lite import SurrealConfigDict


class Post(BaseSurrealModel):
    model_config = SurrealConfigDict(server_fields=["created_at"])
    id: str
    title: str = ""
    created_at: Any = None


print("server-owned columns:", Post.get_server_fields())

server-owned columns: frozenset({'created_at'})


In [11]:
await client.query("REMOVE TABLE IF EXISTS Post;", {})
await client.query(
    "DEFINE FIELD created_at ON Post TYPE option<datetime> DEFAULT time::now();", {}
)

post = await Post(id="p1", title="First").save()
print("created_at, hydrated straight back onto the instance:", post.created_at)

created_at, hydrated straight back onto the instance: 2026-09-12 12:45:50.564023+00:00


### Creates omit it, replaces keep it — and the server is why

`_write_payload()` excludes `server_fields` from a **create**, because omission is precisely how
the `DEFAULT` gets to apply. On a **replace** (`update()` / `upsert()`, which compile to
`UPDATE … CONTENT`) omission means something else entirely: a `DEFAULT` is a *create-time*
default on both DB lines, so an omitted optional column is **deleted** and an omitted required
one is a hard server error. The instance's value is carried instead.

In [12]:
print("create  payload:", post._write_payload())
print("replace payload:", post._write_payload(replace=True))

create  payload: {'title': 'First'}
replace payload: {'title': 'First', 'created_at': datetime.datetime(2026, 9, 12, 12, 45, 50, 564023, tzinfo=datetime.timezone.utc)}


In [13]:
stamp = (await raw_row("Post", "p1"))["created_at"]

post.title = "Renamed"
await post.update()          # a full replace

print("title:      ", (await raw_row("Post", "p1"))["title"])
print("created_at: ", (await raw_row("Post", "p1"))["created_at"])
print("stamp survived the replace:", (await raw_row("Post", "p1"))["created_at"] == stamp)

title:       Renamed
created_at:  2026-09-12 12:45:50.564023+00:00
stamp survived the replace: True


### An explicit write is still allowed

This is where `server_fields` differs from a computed field. A `Computed[...]` field is defined
by `DEFINE FIELD … VALUE`, so the server *discards* any client write — the ORM raises rather
than let an invisible no-op through. A `server_fields` entry is merely a column the ORM is told
not to volunteer, so naming it is taken as consent: a backfill, an admin correction.

In [14]:
from datetime import UTC, datetime

await post.merge(created_at=datetime(2020, 1, 1, tzinfo=UTC))
print("explicitly backfilled:", (await raw_row("Post", "p1"))["created_at"])

explicitly backfilled: 2020-01-01 00:00:00+00:00


In [15]:
from surreal_orm_lite.functions import Computed, computed


class Article(BaseSurrealModel):
    model_config = SurrealConfigDict(server_fields=["created_at"])
    id: str
    first: str = ""
    created_at: Any = None
    shouted: Computed[str] = computed("string::uppercase(first)")


print("both halves are server-owned:", Article.get_server_fields())

try:
    await Article(id="a").merge(shouted="nope")
except ValueError as error:
    print("computed field refuses the write:")
    print(" ", error)

both halves are server-owned: frozenset({'shouted', 'created_at'})
computed field refuses the write:
  merge(): shouted is a computed field on Article, set server-side by DEFINE FIELD … VALUE and not writable. Update the fields the expression reads instead.


## 6. `merge(refresh=False)` — skip the resync

By default `merge()` re-reads the record afterwards so the instance matches the server. That is
a second round-trip. For a fire-and-forget update — a presence ping, a counter bump — you can
skip it: the write still happens, and the literal keyword arguments are applied locally.

In [16]:
calls: list[str] = []
original_select = client.select


async def counting_select(*args, **kwargs):
    calls.append("select")
    return await original_select(*args, **kwargs)


client.select = counting_select

await ada.merge(display="Ada L.")                    # default: resyncs
print("SELECTs issued with the default:", len(calls))

calls.clear()
await ada.merge(display="Ada Lovelace", refresh=False)
print("SELECTs issued with refresh=False:", len(calls))
print("instance still updated locally:", ada.display)
print("and it really persisted:", (await raw_row("Account", "ada"))["display_name"])

client.select = original_select

SELECTs issued with the default: 1
SELECTs issued with refresh=False: 0
instance still updated locally: Ada Lovelace
and it really persisted: Ada Lovelace


**What you give up.** The "no rows came back ⇒ record not found" check *is* the returned row.
Under `refresh=False` a merge against a record that no longer exists is a silent no-op instead
of a `SurrealDbError`, and a field computed by `server_values` keeps its stale value until the
next `refresh()`. Keep the default whenever you need to know the write landed.

In [17]:
from surreal_orm_lite.exceptions import SurrealDbError

ghost = Account(id="does-not-exist", password="x")

try:
    await ghost.merge(display="anything")
except SurrealDbError as error:
    print("default: the missing record is reported ->", error)

await ghost.merge(display="anything", refresh=False)
print("refresh=False: no error, silently a no-op")

default: the missing record is reported -> Can't refresh data, no record found.
refresh=False: no error, silently a no-op


On the `server_values` path the saving is made differently: the statement is compiled with
`RETURN NONE`, so the server does not send the row back at all.

In [18]:
from surreal_orm_lite.functions import SurrealFunc

await ada.merge(server_values={"display": SurrealFunc("string::uppercase('ada')")})
print("default syncs the computed value:", ada.display)

await ada.merge(
    logins=99,
    server_values={"display": SurrealFunc("string::lowercase('ADA')")},
    refresh=False,
)
print("refresh=False — literal kwarg applied locally:", ada.logins)
print("...but the server-computed field is stale:  ", ada.display)
await ada.refresh()
print("...until you refresh:                        ", ada.display)

default syncs the computed value: ADA
refresh=False — literal kwarg applied locally: 99
...but the server-computed field is stale:   ADA
...until you refresh:                         ada


## 7. Cleanup

In [19]:
for table in ("Account", "Post", "Article"):
    await client.query(f"REMOVE TABLE IF EXISTS {table};", {})

await SurrealDBConnectionManager.close_connection()
print("Cleaned up.")

Cleaned up.
